# Sales Lead Conversion Prediction

**Classification Project 29 of 100** — Predict whether a sales lead will convert to a paying customer.

EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.


## 2. Project Overview

Predict whether a sales lead will convert using acquisition source, demographics, and engagement features. Lead scoring is fundamental to sales operations.

Binary classification with mixed feature types and high-cardinality categorical variables.


## 3. Learning Objectives

1. Lead scoring as a classification problem
2. Handling high-cardinality categorical features
3. Missing value strategies for CRM-style data
4. Feature importance for actionable sales insights
5. LazyPredict and FLAML on real-world marketing data
6. Business value of lead prioritization


## 4. Problem Statement

> Given a lead's source, demographics, and activity data, predict whether they will convert.

Binary classification. F1 and ROC-AUC are primary metrics.


## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Sales teams | Focus on high-probability leads |
| Marketing | Optimize lead acquisition channels |
| Revenue ops | Better pipeline forecasting |
| ML learners | CRM data + high-cardinality features |


## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | ashydv/leads-dataset |
| Features | Lead source, last activity, city, occupation, page views, time on site |
| Target | Converted (0/1) |
| Balance | Moderately imbalanced |


## 7. Dataset Source and License Notes

- Kaggle: ashydv/leads-dataset
- Education/ed-tech lead dataset
- License: Check Kaggle page.


## 8. Environment Setup


In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## 9. Imports


In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")


## 10. Configuration / Constants


In [ ]:
KAGGLE_SLUG = "ashydv/leads-dataset"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 90; MAX_ROWS = 50000
HIGH_CARD_THRESHOLD = 20
print(f"Kaggle slug: {KAGGLE_SLUG}")


## 11. Dataset Download or Loading


In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


## 12. Data Validation Checks


In [ ]:
print(f"Missing values (top 10):\n{df_raw.isnull().sum().sort_values(ascending=False).head(10)}\n")
df = df_raw.copy()
target_candidates = [c for c in df.columns if any(h in c.lower() for h in ['converted','conversion','status','target'])]
TARGET = target_candidates[0] if target_candidates else df.columns[-1]
print(f"Target column: {TARGET}")
id_cols = [c for c in df.columns if c.lower() in ['prospect id','lead number','id','index','unnamed: 0']]
if id_cols: df = df.drop(columns=id_cols); print(f"Dropped: {id_cols}")
df = df.replace('Select', np.nan)
high_miss = [c for c in df.columns if df[c].isnull().mean() > 0.6]
if high_miss: df = df.drop(columns=high_miss); print(f"Dropped high-missing: {len(high_miss)}")
dupes = df.duplicated().sum()
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f"\nClean shape: {df.shape}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")


## 13. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=['steelblue','coral'])
ax.set_title(f'Target Distribution: {TARGET}')
plt.tight_layout(); plt.show()


In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
if len(num_feats) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    for ax, col in zip(axes.flat, num_feats[:4]):
        for val in sorted(df[TARGET].unique()):
            ax.hist(df[df[TARGET]==val][col].dropna(), bins=25, alpha=0.4, label=f'{TARGET}={val}')
        ax.set_title(col); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


In [ ]:
cat_feats = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_feats:
    for col in cat_feats[:5]:
        print(f"{col}: {df[col].nunique()} unique")
print(f"\nNumeric: {len(num_feats)}, Categorical: {len(cat_feats)}")


## 14. Target Analysis

Lead conversion datasets often have 30-40% positive rate.


In [ ]:
print(df[TARGET].value_counts(normalize=True))
print(f"\nConversion rate: {df[TARGET].mean():.1%}")


## 15. Train / Validation / Test Split


In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), name=TARGET)
cat_cols_all = X.select_dtypes(include=['object','category']).columns.tolist()
high_card = [c for c in cat_cols_all if X[c].nunique() > HIGH_CARD_THRESHOLD]
if high_card:
    X = X.drop(columns=high_card)
    print(f"Dropped high-cardinality: {high_card}")
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")


## 16. Preprocessing Strategy

- Numeric: impute median, standardize
- Categorical: impute 'Unknown', OHE (max_categories=15)


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols))
if cat_cols: transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=15))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')


## 17. Feature Engineering

Create engagement intensity features.


In [ ]:
def engineer_features(d):
    d = d.copy()
    nc = d.select_dtypes(include=[np.number]).columns
    page_cols = [c for c in nc if any(k in c.lower() for k in ['page','visit','view'])]
    time_cols = [c for c in nc if any(k in c.lower() for k in ['time','duration','spent'])]
    if page_cols and time_cols:
        d['ENGAGEMENT_SCORE'] = d[page_cols[0]].fillna(0) * d[time_cols[0]].fillna(0)
    return d
X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_cols))
if cat_cols: transformers.append(('cat', Pipeline([('imp',SimpleImputer(strategy='constant',fill_value='Unknown')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False,max_categories=15))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')


## 18. Baseline Model


In [ ]:
results = {}
for name, clf in [('Dummy', DummyClassifier(strategy='most_frequent', random_state=SEED)),
                  ('LogReg', LogisticRegression(max_iter=1000, random_state=SEED)),
                  ('RF', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]:
    pipe = Pipeline([('pre',preprocessor),('clf',clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {'Accuracy': accuracy_score(y_val,yp), 'F1': f1_score(y_val,yp,zero_division=0)}
    if hasattr(clf, 'predict_proba'):
        try: r['ROC-AUC'] = roc_auc_score(y_val, pipe.predict_proba(X_val)[:,1])
        except: r['ROC-AUC'] = np.nan
    results[name] = r; print(f'{name}: {r}')


## 19. LazyPredict Benchmark


In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)


## 20. FLAML AutoML Run


In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='classification', metric='f1',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'Accuracy': accuracy_score(y_val, yp_fl), 'F1': f1_score(y_val, yp_fl, zero_division=0)}
print(results['FLAML'])


## 21. Top 3 Model Selection


In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBClassifier; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm: top3['LightGBM'] = LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3['ExtraTrees'] = ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb: top3['XGBoost'] = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1, eval_metric='logloss')
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3['AdaBoost'] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')


## 22. Final Training and Evaluation of Top 3


In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
labels = ['Not Converted', 'Converted']
final = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t); ypr = mdl.predict_proba(X_te_t)[:,1]
    final[name] = {'Accuracy': accuracy_score(y_test,yp), 'Precision': precision_score(y_test,yp,zero_division=0),
                   'Recall': recall_score(y_test,yp,zero_division=0), 'F1': f1_score(y_test,yp,zero_division=0),
                   'ROC-AUC': roc_auc_score(y_test,ypr)}
    print(f'\n{name}:'); print(classification_report(y_test,yp,target_names=labels))
pd.DataFrame(final).T


In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_te_t), ax=ax, cmap='Blues', display_labels=labels)
    ax.set_title(n)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m, X_te_t, y_test, ax=ax, name=n)
ax.plot([0,1],[0,1],'k--',alpha=0.5); ax.set_title('ROC Curves')
plt.tight_layout(); plt.show()


## 23. Error Analysis


In [ ]:
bm = list(top3.values())[0]; ypb = bm.predict(X_te_t)
fn = (y_test.values==1) & (ypb==0); fp = (y_test.values==0) & (ypb==1)
print(f'False Negatives (missed conversions): {fn.sum()}')
print(f'False Positives (wasted effort): {fp.sum()}')
print(f'Total misclassified: {(y_test.values!=ypb).sum()}/{len(y_test)} ({(y_test.values!=ypb).mean():.1%})')


## 24. Interpretation and Business Insight

- **Lead source matters**: Referrals convert higher
- **Engagement signals**: Time on site + page views predict conversion
- **Last activity type**: Recent interactions indicate hot leads
- **Sales efficiency**: Score top 30% for immediate follow-up


## 25. Limitations

1. CRM data quality — many 'Select' placeholders
2. No temporal information
3. High cardinality limits encoding
4. Binary conversion doesn't capture partial engagement
5. Potential selection bias


## 26. How to Improve This Project

1. Add lead response time
2. Use target encoding for high-cardinality
3. Build lead scoring dashboard
4. Add sequential activity features
5. A/B test model-prioritized assignment


## 27. Production Considerations

- Real-time scoring in CRM
- Score refresh as activity updates
- Integration with Salesforce/HubSpot
- Calibrated probabilities for lead grading
- Weekly model retraining


## 28. Common Mistakes

1. Not handling 'Select' placeholder as NaN
2. Including ID columns
3. OHE with 100+ city categories
4. Lead source as potential leakage
5. Ignoring temporal patterns


## 29. Mini Challenge / Exercises

1. Target encoding for Lead Source
2. Create lead score A/B/C/D from probabilities
3. Conversion rate in top 20% of scores?
4. Remove page views — model degrades?
5. Rule-based classifier vs ML


## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary — lead conversion |
| Dataset | Leads dataset with CRM features |
| Difficulty | Medium |
| Key insight | Engagement + source drive conversion |

Lead scoring pipeline for sales optimization, handling messy CRM data.
